Question:

Explain how a Git branching strategy resolves this coordination problem without
slowing the team down. Describe the exact sequence of Git commands each team member
would use — from creating their own branch through to raising a Pull Request and completing a
reviewed merge. Then describe one realistic scenario in this project where a merge conflict
would occur, explain what causes it at the file level, and walk through how you would resolve it
using the VS Code merge tool.

The problem

Three people are working on the same project:

Person 1 → Data Cleaning
Person 2 → Feature Engineering
You      → Model Building

If everyone works directly on main:

             main
              │
       ┌──────┼──────┐
       ↓      ↓      ↓
    Cleaning Features Model
       ↓      ↓      ↓
      BREAKING EACH OTHER'S WORK ❌

This is the problem.

The solution: Branches

Instead of everyone working directly on main, each person creates their own branch.

                 main
                  │
       ┌──────────┼──────────┐
       ↓          ↓          ↓
data-cleaning  feature-    model-
               engineering development

Now everyone can work independently.

Teammate 1

Works on:

data-cleaning
Teammate 2

Works on:

feature-engineering
You

Work on:

model-development

Their changes don't immediately affect main.

Why does this solve the problem?

Suppose you are building the model and accidentally introduce a bug.

Your branch becomes:

model-development → has bug ❌

But:

main → still working ✅

So the manager's requirement is satisfied:

The main branch stays stable and working.

Before changes enter main, they go through a Pull Request (PR) and code review.

Developer branch
       ↓
    Push code
       ↓
 Pull Request
       ↓
 Code Review
       ↓
 Approved? ✅
       ↓
 Merge into main

Step 1 — Start from main

Before creating a branch, everyone should make sure their local main is up to date.

git switch main
git pull origin main
What do these do?
git switch main

➡️ Move to the main branch.

git pull origin main

➡️ Download the latest changes from GitHub's main branch.

So now everyone starts from the latest working version.

Step 2 — Create your own branch
👨‍💻 Teammate 1 — Data Cleaning
git switch -c data-cleaning

This:

creates a new branch called data-cleaning
immediately switches to it
👩‍💻 Teammate 2 — Feature Engineering
git switch -c feature-engineering
👨‍💻 You — Model Building
git switch -c model-development

Now:

main
 │
 ├── data-cleaning
 ├── feature-engineering
 └── model-development

Everyone can work independently.

Step 3 — Write your code

Now each person works on their assigned task.

For example:

Data scientist 1

Changes:

data_cleaning.py
Data scientist 2

Changes:

feature_engineering.py
You

Changes:

model.py

Step 4 — Check what changed

After finishing your work:

git status

This shows which files have been modified.

For example:

modified: model.py
Step 5 — Add the changes
git add .

This stages the changes for the commit.

You can think:

Working files
     ↓
git add .
     ↓
Staged files
Step 6 — Commit
git commit -m "Build churn prediction model"

A commit is basically a saved checkpoint of your changes.

For example:

git commit -m "Clean missing order data"

for the data-cleaning teammate.

And:

git commit -m "Add customer churn features"

for the feature-engineering teammate.

Step 7 — Push your branch to GitHub

For your first push:

git push -u origin model-development

The others would use:

git push -u origin data-cleaning

or:

git push -u origin feature-engineering

Now the branch exists on GitHub.

Step 8 — Create Pull Request

Now we do not directly merge into main.

On GitHub, create a:

Pull Request (PR)

For example:

model-development
        ↓
Pull Request
        ↓
main

The PR basically says:

"I have completed my work. Please review my changes before adding them to main."

Step 9 — Code Review

Another teammate reviews the code.

They check:

Is the code correct?
Does it break anything?
Does it follow the project standards?
Are there unnecessary changes?
Do the tests pass?

If there is a problem, the reviewer can comment:

"Please fix this section."

You make the correction:

git add .
git commit -m "Fix model validation"
git push

The same Pull Request gets updated automatically.

Step 10 — Approval and Merge

Once the reviewer approves the PR:

Pull Request
     ↓
Review
     ↓
Approved ✅
     ↓
Merge into main

The merge can be done through GitHub.

After merging, everyone should update their local main:

git switch main
git pull origin main
🎯 Complete workflow

This is the important sequence to remember:

git switch main
        ↓
git pull origin main
        ↓
git switch -c my-branch
        ↓
Write code
        ↓
git status
        ↓
git add .
        ↓
git commit -m "message"
        ↓
git push -u origin my-branch
        ↓
Create Pull Request
        ↓
Code Review
        ↓
Fix issues if required
        ↓
Approved
        ↓
Merge into main
        ↓
git switch main
git pull origin main

1. What is a merge conflict?

A merge conflict happens when two branches change the same part of the same file in different ways, and Git cannot automatically decide which change should be kept.

In our project:

Teammate 2 → feature-engineering branch
You        → model-development branch

Suppose both of you edit the same file:

model.py
2. Realistic example from our project

Suppose the original model.py contains:

features = ["age", "order_count"]
Your teammate changes it

They add a feature:

features = ["age", "order_count", "avg_order_value"]
You change the same line

You add different features:

features = ["age", "order_count", "days_since_last_order"]

Now Git sees:

Original
    ↓
features = ["age", "order_count"]


       ↙                    ↘


Feature branch          Model branch
       ↓                     ↓
avg_order_value       days_since_last_order

Git doesn't know:

"Should I keep avg_order_value or days_since_last_order?"

So Git creates a merge conflict.

3. What causes the conflict at file level?

Git marks the conflicting section like this:

<<<<<<< HEAD
features = ["age", "order_count", "days_since_last_order"]
=======
features = ["age", "order_count", "avg_order_value"]
>>>>>>> feature-engineering

Don't worry about remembering these symbols individually.

They basically mean:

<<<<<<< HEAD
Your/current branch's version
=======
Other branch's version
>>>>>>> other branch

Git is saying:

"I found two different versions of this same section. You decide which one should remain."

4. Open the conflict in VS Code

When you open model.py in VS Code, you'll see the conflicting section.

VS Code normally gives options such as:

Accept Current Change
Accept Incoming Change
Accept Both Changes
Compare Changes
Accept Current Change

Keep your branch's version.

features = ["age", "order_count", "days_since_last_order"]
Accept Incoming Change

Keep the other branch's version.

features = ["age", "order_count", "avg_order_value"]
Accept Both Changes

Keep both.

You might end up with:

features = [
    "age",
    "order_count",
    "days_since_last_order",
    "avg_order_value"
]

In our situation, Accept Both Changes may actually make sense because both features could be useful for churn prediction.

But we shouldn't blindly choose it—we should check whether both features are valid and whether the model expects them.

5. Manually resolve if necessary

Sometimes neither option is exactly what we want.

We can manually edit the code.

For example:

features = [
    "age",
    "order_count",
    "avg_order_value",
    "days_since_last_order"
]

Then make sure the conflict markers are completely removed:

<<<<<<<
=======
>>>>>>>

They must not remain in the final code.

6. Save and check the file

After resolving the conflict in VS Code:

git status

Git will tell you that model.py has been resolved/staged or remains unmerged depending on what you have done.

Then:

git add model.py

This tells Git:

"I have resolved this conflict."

Then commit:

git commit -m "Resolve merge conflict in model features"

And push if you're resolving it on your branch:

git push

If the conflict occurred while updating your branch from main, the exact final push/PR flow depends on where the merge was performed, but the core resolution is the same: edit → stage → commit → push.

🧠 Complete conflict process

Remember this:

Two developers edit same code
          ↓
Git cannot decide
          ↓
MERGE CONFLICT
          ↓
Open file in VS Code
          ↓
See conflict markers
          ↓
Choose:
Current / Incoming / Both
          ↓
OR manually edit
          ↓
Remove conflict markers
          ↓
Test the code
          ↓
git add
          ↓
git commit
          ↓
git push